# Step 3: real training (Overture, then case2s)
§1–8: Overture train → val only; no test split here (the single test run comes after the pooling × seeds runs are compared). `evaluate`, `is_better` and `broken_run_problems` live in `training.py` (moved there 2026-10-23, so `03_siamese_analysis.ipynb` can import them); each one's unit tests stay here, under its section. `run_training` stays in this notebook (it reads the loaders built in §2). §9–11: the same model and protocol retrained on the `case2s` dataset (Step 4 prep; was `02_siamese_training.ipynb §9–11`). Stage-by-stage build and tests: `01_siamese_build.ipynb`.

**Spec (decided 2026-10-21)**

| Row | Decision |
|---|---|
| Data | Overture only (train → val); Stamford dropped from Step 3; case2 left as is (172 train rows, no weighting) |
| Model | `SiameseMatcher(Encoder(...))` from `model.py`: BiLSTM 1 × 128, emb 32, max length 85; pooling mean or max |
| Optimiser | Adam lr 1e-3, clip 1.0, batch 64, `BCEWithLogitsLoss` (no `pos_weight`) |
| Threshold | 0.5 (same as 1b) |
| Watched each epoch | train loss, val loss, val non-identical found / case1 / easy |
| Best epoch | highest val non-identical found **among epochs with val case1 ≥ 98%**; tie → lower val loss |
| Stop | 3 epochs without a new best (`PATIENCE`), or 20 epochs (`MAX_EPOCHS`) |
| No epoch qualifies | keep the last epoch's weights, `best = None` (patience then ends the run after 3 epochs) |
| Runs | mean vs max × seeds 0, 1, 2 = 6 runs (§8); "better" judged on mean ± spread over seeds |
| Broken-run checks | untrained val loss ≈ 0.693; train loss parked at ≈ 0.325 = learned only the base rate; no NaN; val found > 0% |
| Saved per run | best `state_dict` + vocab + config + seed + best row → `checkpoints/overture_siamese_seed_{seed}_pooling_{pool}.pt`; history → `checkpoints/history_seed_{seed}_pooling_{pool}.csv` |
| Reference (val) | LogReg 1b: found 81.8 · RF: found 85.4 · case1 99.5–99.8 · easy 100.0. Prediction (David): easy < 98% (no ZIP) |

| § | What | Where |
|---|---|---|
| 1 | Truncation count | 6 train / 0 val strings > 85 chars (Diagnostics) |
| 2 | Datasets + loaders | `train_loader` (shuffled), `val_loader` (not shuffled) |
| 3 | `evaluate` (+ test) | val loss + found / case1 / easy |
| 4 | Timing | ≈ 4.1 min/epoch (Diagnostics) |
| 5 | `is_better` (+ tests) | early-stopping rule |
| 6 | `run_training` | one run with patience, returns history / best / weights |
| 7 | `broken_run_problems` (+ tests) | four checks on a run's history; [] = healthy |
| 8 | Experiment queue + summary | 6 runs, each saved when it finishes, finished runs skipped on re-run; summary open |
| 9–11 | case2s retrain (Step 4 prep) | one run, mean / seed 0 → `checkpoints_case2s/` |

In [1]:
%load_ext autoreload
%autoreload 2

import copy
import os
import sys
import time
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd
import numpy as np
import torch
from torch import nn
import itertools
from torch.utils.data import DataLoader, Subset

from helpers.pairs import load_pairs
from helpers.check_model_performance import review_typical_slices
import math

from siamese_model.model import build_vocab, Encoder, SiameseMatcher, PairDataset, make_collate
from siamese_model.training import (train_epoch, predict, evaluate, is_better, broken_run_problems, train_run,
                                    load_run, rebuild_model, THRESHOLD, CASE1_GUARDRAIL, CHECKPOINT_DIR)

pd.set_option("display.max_rows", None)        # all rows
pd.set_option("display.max_columns", None)     # all columns
pd.set_option("display.max_colwidth", None)    # full text inside each cell (long address strings)
pd.set_option("display.width", None)           # don't wrap/clip the table at a fixed character width

In [2]:
SEED = 0
BATCH_SIZE = 64
LR = 1e-3
N_TIMING_BATCHES = 200      # §4: time this many batches, extrapolate to a full epoch
MAX_EPOCHS = 20           # §4: ~4.1 min/epoch -> worst case ~83 min (Diagnostics)
PATIENCE = 3
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# THRESHOLD, CASE1_GUARDRAIL, CHECKPOINT_DIR: from training.py (setup cell)


In [3]:
ot_train, ot_train_label = load_pairs("overture", "train")
ot_val, ot_val_label = load_pairs("overture", "val")
vocab = build_vocab(ot_train)          # train split only
print(len(ot_train), len(ot_val), len(vocab))

202568 41865 40


## 1. Truncation count
Strings longer than 85 characters before `normalize` cuts them, per column and split. Result in Diagnostics.

In [4]:
print(len(ot_train[ot_train["address_a"].str.len() > 85]))
print(len(ot_train[ot_train["address_b"].str.len() > 85]))
print(len(ot_val[ot_val["address_a"].str.len() > 85]))
print(len(ot_val[ot_val["address_b"].str.len() > 85]))

# print(ot_train[ot_train["address_a"].str.len() > 85]["address_a"])
print(ot_train[ot_train["address_b"].str.len() > 85][["address_b", "neg_type"]])

6
6
0
0
                                                                                                                        address_b  \
18026   WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
130983  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
162764  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
195747  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
198904  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   
238879  WHETTEN GRADUATE CENTER SECOND FLOOR UNIVERSITY OF CONNECTICUT 438 WHITNEY RD EXTENSION UNIT-1152 STORRS 06269-1152 UNITE   

       neg_type  
18026     case0  
130983    case3  
162764

## 2. Datasets and loaders
Train shuffled; val **not** shuffled (`predict` returns probabilities in loader order). `run_training` reads these global loaders, so no other cell may rebuild them.

In [5]:
train_dataset = PairDataset(ot_train, ot_train_label)
val_dataset = PairDataset(ot_val, ot_val_label)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle = True, collate_fn = make_collate(vocab))
val_loader = DataLoader(val_dataset, batch_size=64, shuffle = False, collate_fn= make_collate(vocab))

## 3. `evaluate` (in `training.py`)
One model's val numbers as a dict: `{"val_loss", "found", "case1", "easy"}` (% correct on those slices, same slices as `results.md`). The last lines are the untrained-model check (Diagnostics).

In [6]:
# evaluate lives in training.py; the untrained-model check (Diagnostics) stays here
evaluation_model = SiameseMatcher(Encoder(len(vocab)))
untrained = evaluate(evaluation_model, val_loader, ot_val, ot_val_label)
print(untrained)

{'val_loss': 0.7024561166763306, 'found': 99.12170639899624, 'case1': 0.4498947826717945, 'easy': 0.5489670200729162}


In [7]:
# Unit test for evaluate, on the untrained model's result from the cell above (no extra val pass)
assert set(untrained) == {"val_loss", "found", "case1", "easy"}
assert all(0 <= untrained[k] <= 100 for k in ["found", "case1", "easy"])
assert math.isfinite(untrained["val_loss"])
assert all(type(v) is float for v in untrained.values())
assert abs(untrained["val_loss"] - 0.693) < 0.05

## 4. Timing
Times `N_TIMING_BATCHES` training batches (extrapolated to a full epoch) and one val pass, on a throwaway model. Result and the `MAX_EPOCHS` / `PATIENCE` decision in Diagnostics.

In [8]:
torch.manual_seed(SEED)
timing_model = SiameseMatcher(Encoder(len(vocab)))
timing_opt = torch.optim.Adam(timing_model.parameters(), lr=LR)
subset_loader = DataLoader(Subset(train_dataset, range(N_TIMING_BATCHES * BATCH_SIZE)), batch_size=BATCH_SIZE, collate_fn = make_collate(vocab))

# train/val loaders come from §2 (train shuffled); don't rebuild them here

t0 = time.perf_counter()
train_epoch(timing_model, subset_loader, timing_opt, nn.BCEWithLogitsLoss())
train_seconds = time.perf_counter() - t0
print(f"train: {train_seconds / N_TIMING_BATCHES:.3f} s/batch -> {train_seconds / N_TIMING_BATCHES * len(train_loader) / 60:.1f} min/epoch")

t0 = time.perf_counter()
predict(timing_model, val_loader)
print(f"val pass: {time.perf_counter() - t0:.1f} s")

train: 0.064 s/batch -> 3.4 min/epoch
val pass: 10.5 s


## 5. `is_better(row, best)` (in `training.py`)
The early-stopping rule: a row below `CASE1_GUARDRAIL` never qualifies; the first qualifying row is best; then higher `found` wins, a tie goes to the lower `val_loss`.

In [9]:
# Unit tests for is_better (hand-made rows; Claude wrote these at David's request, 2026-10-22)
r = lambda found, case1=99.0, val_loss=0.1: {"found": found, "case1": case1, "val_loss": val_loss}

assert is_better(r(100, case1=0), None) is False         # below guardrail, nothing yet (the untrained model's profile)
assert is_better(r(50), None) is True                    # first qualifying row
assert is_better(r(60), r(50)) is True                   # higher found
assert is_better(r(90, case1=97), r(50)) is False        # higher found but below guardrail
assert is_better(r(50, val_loss=0.1), r(50, val_loss=0.2)) is True    # tie on found -> lower val loss
assert is_better(r(50, val_loss=0.2), r(50, val_loss=0.1)) is False   # tie on found, higher val loss
assert is_better(r(40), r(50)) is False                  # lower found
assert is_better(r(50, case1=98.0), None) is True        # exactly at the guardrail counts
print("is_better: all tests pass")

is_better: all tests pass


## 6. `run_training`
One full run: seed → model → epochs of `train_epoch` + `evaluate`; keeps a deep copy of the best epoch's weights; stops after `PATIENCE` epochs without a new best. If no epoch qualifies, returns the last epoch's weights with `best = None`.

In [10]:
def run_training(seed=SEED, pooling="mean"):
    torch.manual_seed(seed)
    model = SiameseMatcher(Encoder(len(vocab),pooling = pooling))
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr = LR)
    loss_fn = nn.BCEWithLogitsLoss()

    best, best_state, epochs_since_best = None, None, 0
    history = []
    for epoch in range(MAX_EPOCHS):
        train_loss = train_epoch(model, train_loader, opt, loss_fn)

        row = {"epoch": epoch, "train_loss": train_loss, **evaluate(model, val_loader, ot_val, ot_val_label)}
        print(row)
        history.append(row)

        if is_better(row, best):
            best, best_state, epochs_since_best = row, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_since_best += 1
            if epochs_since_best >= PATIENCE:
                break
    if best is None:                                      # no epoch reached the case1 guardrail -> keep the last epoch's weights
        best_state = copy.deepcopy(model.state_dict())
    return pd.DataFrame(history), best, best_state

## 7. Broken-run checks (`broken_run_problems` in `training.py`)
A function `broken_run_problems(history)` → list of problems (empty = healthy), called on each run's history by the queue (§8), so it is defined first:

| Check | Rule |
|---|---|
| No NaN | `history.isna().any().any()` is False |
| Beats a coin flip after 1 epoch | `history["val_loss"].iloc[0] < 0.693` |
| Not stuck at the base rate | `history["train_loss"].min() < 0.30` |
| Finds something | `history["found"].max() > 0` |

Test it on the broken run's history (2026-10-21, Diagnostics): it must return at least one problem.

In [11]:
# broken_run_problems lives in training.py (Claude wrote it at David's request, 2026-10-22); tests stay here
# Tests: a known-broken run must be flagged, a healthy one must not
broken = pd.read_csv("broken/history_mean_seed0.csv")
assert broken_run_problems(broken) != []
print("broken run:", broken_run_problems(broken))          # expected: only the epoch-0 val loss check fires

healthy = pd.DataFrame({"train_loss": [0.05, 0.02], "val_loss": [0.03, 0.02], "found": [78.0, 80.0]})
assert broken_run_problems(healthy) == []

assert broken_run_problems(healthy.assign(val_loss=[float("nan"), 0.02])) != []       # NaN
assert broken_run_problems(healthy.assign(train_loss=[0.33, 0.32])) != []             # stuck at the base rate
assert broken_run_problems(healthy.assign(found=[0.0, 0.0])) != []                    # finds nothing
print("broken_run_problems: all tests pass")


broken run: ['val loss after epoch 0 = 0.992 (coin flip = 0.693)']
broken_run_problems: all tests pass


## 8. Experiment queue: pooling × seeds
6 runs back to back (worst case ≈ 8.3 h). Each run is saved as soon as it finishes; re-running the cell skips runs already saved and carries on.

**To start:** restart the kernel, run every cell from the top through §7, keep the Mac awake (`caffeinate -dims` in a terminal), run this cell, and watch epoch 0 of `mean` / seed 0: expect val loss well below 0.693 and found > 0 (the first shuffled run had found ≈ 79, val loss ≈ 0.02).

In [12]:
for pool_mode, seed in itertools.product(["mean", "max"], [0,1,2]):
    if os.path.exists(f"checkpoints/overture_siamese_seed_{seed}_pooling_{pool_mode}.pt"):
        continue

    history, best, best_state = run_training(seed = seed, pooling = pool_mode)
    
    history.to_csv(
    os.path.join(CHECKPOINT_DIR, f"history_seed_{seed}_pooling_{pool_mode}.csv"), index = False
    )
    torch.save({
        "state_dict": best_state, "vocab": vocab, 
        "config": {"emb_dim": 32, "hidden": 128, "pooling": pool_mode, "max_len": 85},
        "seed": seed, "best":best}, f"checkpoints/overture_siamese_seed_{seed}_pooling_{pool_mode}.pt")
    print(f"finishing seed:{seed} | pool:{pool_mode} -> best: {best}")
    print(f"problems: {broken_run_problems(history)}")

**Summary (open, after the queue):** one row per run from `torch.load(path)["best"]` → DataFrame; mean ± spread of `found` per pooling. Also reload one checkpoint from its own `config` + `vocab`, `evaluate` it, and check it matches `best` on `val_loss`, `found`, `case1`, `easy` (proves a file alone rebuilds the model).

In [13]:
rows = []
for pool_mode, seed in itertools.product(["mean", "max"], [0,1,2]):
    history, ckpt = load_run(seed, pool_mode)

    rows.append({"pooling": pool_mode, "seed": seed, **(ckpt["best"] or {}), "epochs_run": len(history), "problems": broken_run_problems(history)})

    # reload check: the file alone rebuilds the model that produced `best` (skipped when no epoch qualified)
    if ckpt["best"] is None:
        continue
    reloaded = rebuild_model(ckpt)
    after = evaluate(reloaded, val_loader, ot_val, ot_val_label)
    for k in ["val_loss", "found", "case1", "easy"]:
        assert after[k] == ckpt["best"][k], (k, after[k], ckpt["best"][k])

summary = pd.DataFrame(rows)
display(summary)
summary.groupby("pooling")[["found", "case1", "easy"]].agg(["mean", "min", "max"])


KeyboardInterrupt: 

## 9. Step 4 prep: train the siamese model on `case2s`

**Why:** the Step 3 model saw only easy and same-street negatives, so its `case2s` score (in `model_baseline/07_backtest_case2s.ipynb`) can't be compared with the classical models, which were retrained on the new train split. §9–11 train the same model on `case2s`, so the backtest compares like with like.

(Was `02_siamese_training.ipynb §9–11`.) Its data, vocab and loaders are prefixed `c2_`, so §1–8's `ot_*`, `vocab` and loaders stay intact.

**What it does:** the same protocol as §1–8 (character-level BiLSTM encoder, mean and max pooling (max added 2026-11-18), seed 0, Adam 1e-3, up to 20 epochs, patience 3, best epoch = highest val "found" among epochs above the case1 guardrail), now on the `case2s` train and val splits. One run per pooling. Checkpoint and history go to `checkpoints_case2s/` (git-ignored), same file names as `checkpoints/`, so `load_run(0, "mean", "checkpoints_case2s")` loads it.

**Time:** about 4 min per epoch on the old train split; the `case2s` train split has the same number of rows. Worst case 20 epochs ≈ 80 min per run. A run is skipped if its checkpoint already exists.

**Decision for you:** the early-stopping rule only looks at non-identical matches, case1 and easy negatives, not at `case2s`. Kept identical to Step 3 so the results are comparable; changing it (for example adding a `case2s` guardrail) is your call.

In [14]:
# case2s setup: constants and the checkpoint folder (imports: Setup at the top)
DATASET = "overture_case2s"
CHECKPOINT_DIR_CASE2S = "checkpoints_case2s"   # git-ignored, relative to siamese_model/
SEED = 0
POOLINGS = ["mean", "max"]   # mean = the original case2s run; max added 2026-11-18 (runs already saved are skipped)
BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE = 64, 1e-3, 20, 3
os.makedirs(CHECKPOINT_DIR_CASE2S, exist_ok = True)

## 10. case2s: load and build the vocabulary
**Why:** the model reads characters, so it needs a character → id table. As in Step 3 it is built from the train split only, so characters seen only in val / test map to UNK.  
**Does:** loads the `case2s` train and val pairs, builds the vocabulary from train, wraps each split in a `DataLoader` (train shuffled, val not).  
**Output:** `c2_vocab`, `c2_train_loader`, `c2_val_loader`

In [15]:
# Load case2s c2_train / c2_val, c2_vocab from c2_train only, loaders
c2_train, c2_train_label = load_pairs(DATASET, "train")
c2_val, c2_val_label = load_pairs(DATASET, "val")
c2_vocab = build_vocab(c2_train)
print(len(c2_train), len(c2_val), len(c2_vocab))

c2_train_loader = DataLoader(PairDataset(c2_train, c2_train_label), batch_size = BATCH_SIZE, shuffle = True, collate_fn = make_collate(c2_vocab))
c2_val_loader = DataLoader(PairDataset(c2_val, c2_val_label), batch_size = BATCH_SIZE, shuffle = False, collate_fn = make_collate(c2_vocab))

202568 41865 40


## 11. case2s: train and save
**Why:** one run with the Step 3 settings, kept on disk so the backtest can load it and this cell is skipped on a re-run.  
**Does:** `train_run` (§6's loop, as a function in `training.py`) trains until patience runs out, keeping the best epoch's weights; the history and the checkpoint (weights, vocab, config, seed, best row) are saved; `broken_run_problems` lists anything suspicious in the history (`[]` = healthy).  
**Output:** per pooling in `POOLINGS`: `checkpoints_case2s/overture_siamese_seed_0_pooling_{mean,max}.pt`, `history_seed_0_pooling_{mean,max}.csv`; histories in `histories_c2`

In [16]:
# Train each pooling (skipped if its checkpoint exists), save history + checkpoint, check the run
histories_c2 = {}
for POOLING in POOLINGS:
    ckpt_path = f"{CHECKPOINT_DIR_CASE2S}/overture_siamese_seed_{SEED}_pooling_{POOLING}.pt"
    if not os.path.exists(ckpt_path):
        history, best, best_state = train_run(SEED, POOLING, c2_vocab, c2_train_loader, c2_val_loader, c2_val, c2_val_label,
                                              lr = LR, max_epochs = MAX_EPOCHS, patience = PATIENCE)
        history.to_csv(f"{CHECKPOINT_DIR_CASE2S}/history_seed_{SEED}_pooling_{POOLING}.csv", index = False)
        torch.save({
            "state_dict": best_state, "vocab": c2_vocab,
            "config": {"emb_dim": 32, "hidden": 128, "pooling": POOLING, "max_len": 85},
            "seed": SEED, "best": best}, ckpt_path)

    history, ckpt = load_run(SEED, POOLING, checkpoint_dir = CHECKPOINT_DIR_CASE2S)
    histories_c2[POOLING] = history
    print(f"{POOLING}: best epoch: {ckpt['best']}")
    print(f"{POOLING}: problems: {broken_run_problems(history)}")


mean: best epoch: {'epoch': 8, 'train_loss': 0.010607756845280598, 'val_loss': 0.032220665365457535, 'found': 83.06148055207026, 'case1': 99.25259415136783, 'easy': 99.95210936479603}
mean: problems: []
{'epoch': 0, 'train_loss': 0.060193893358986464, 'val_loss': 0.04238458350300789, 'found': 60.97867001254705, 'case1': 99.94920542776286, 'easy': 100.0}
{'epoch': 1, 'train_loss': 0.03137034192251494, 'val_loss': 0.03347691893577576, 'found': 74.40401505646173, 'case1': 99.5646179522531, 'easy': 100.0}
{'epoch': 2, 'train_loss': 0.024472576519531396, 'val_loss': 0.031673334538936615, 'found': 74.40401505646173, 'case1': 99.62266889195269, 'easy': 100.0}
{'epoch': 3, 'train_loss': 0.02088908323614858, 'val_loss': 0.03092069737613201, 'found': 75.78419071518194, 'case1': 99.66620709672738, 'easy': 100.0}
{'epoch': 4, 'train_loss': 0.017968392741458623, 'val_loss': 0.032233692705631256, 'found': 79.29736511919698, 'case1': 99.18003047674334, 'easy': 99.98693891767165}
{'epoch': 5, 'train_l

## Diagnostics
One-off queries and their results (question → result → query). Kept out of the pipeline cells.

- *Truncation (§1), 2026-10-21:* how many strings does the 85-character cut affect? → **6 in train (`address_b`), 0 in val**, all the same Foursquare string (`WHETTEN GRADUATE CENTER … STORRS 06269-1152 UNITE…`; 1 positive + 5 easy negatives). Negligible. The cut-off tail is town + ZIP, i.e. some address strings do contain ZIP/city text (relevant to the open ZIP-text question). Measured on raw lengths (no whitespace collapse; the strings are already cleaned). Query: §1 cell.
- *Timing (§4), 2026-10-21, CPU, batch 64:* how long does a run take, worst case? → train ≈ **204.3 s/epoch** (200 batches timed, × 3,166 / 200 ≈ 0.065 s/batch) + val pass ≈ **43.6 s** ⇒ **≈ 248 s ≈ 4.1 min per epoch**. Worst case = MAX_EPOCHS × 248 s: 15 → ≈ 62 min · **20 → ≈ 83 min** · 50 → ≈ 3.4 h; the follow-up (mean/max × 3 seeds = 6 runs) × 6 (20 → ≈ 8.3 h). Decision: **MAX_EPOCHS = 20, PATIENCE = 3**; CPU is enough, no Colab. Query: §4 cell, separate `perf_counter` timers around `train_epoch` (subset) and `predict(…, val_loader)`.
- *Untrained model on val (§3), 2026-10-21:* → val loss **0.705** (≈ 0.693 ✓), found 100%, case1 0.01%, easy 0.02%: before training every output is ≥ 0.5, so it calls everything a match. Query: last lines of the §3 cell.
- *Broken run, 2026-10-21 (mean, seed 0, 20 epochs):* train loss 0.036 → 0.0008 but val loss **0.99** at epoch 0 (worse than a coin flip), found 0% for epochs 0–6, 37% at best (epoch 18), case1 / easy 100%. Cause (David): training read an unshuffled `train_loader` — an earlier version of the §4 cell rebuilt both loaders without `shuffle=True`, and the train file is stored in order (positives first), so batches were long runs of one label. Fixed: §4 no longer builds loaders. Notes: the in-kernel check `type(train_loader.sampler).__name__` returned `RandomSampler` after the run; the first (shuffled) run on 2026-10-21 had found 78.7 / val loss 0.024 after epoch 0. The broken run's files were moved out of `checkpoints/`. The §7 checks flag it: only `val loss after epoch 0 = 0.992` fires (no NaN, train loss min 0.0008, found max 37). **Resolved 2026-10-23 (David):** his own sampler check confirms the broken run's loader was sequential (supersedes the `RandomSampler` reading above).
